# SAE-Gated RAG: Person 2 latency at batch size 1 (Kaggle)

Fills the two gaps in the Gate 3 / H4 latency table:

1. **Generation latency per condition**: `closed`, `bm25` and `gold`, timed on the *same* questions (those with a gold passage) in one run. Gopal's SageMaker table has closed and BM25-RAG on an L4 but no gold. A Kaggle T4 runs Gemma 3 in fp32 over two GPUs, so its absolute times are not comparable to the L4. This notebook reports the **gold / BM25-RAG ratio** and scales Gopal's L4 BM25-RAG time by it.
2. **BM25 search latency per question**: Pyserini's prebuilt `wikipedia-dpr-100w` index. This is the same 21,015,324-passage DPR Wikipedia corpus as Nidhi's index. Each question is searched alone (one thread), with `k = 3` (what a deployed router fetches) and `k = 100` (what the hole filter fetched). Passage fetching is included. This part is CPU only.

The timing code follows `scripts/sagemaker/run.sh` → `latency_measure.py`: `model.generate` with greedy decoding, up to 32 new tokens, all GPUs synchronised around each call, and 3 warm-up questions per file.

## Setup

1. Attach the **`saerag-p2-bundle`** dataset (the same one the labelling notebook used).
2. Add the `HF_TOKEN` secret (an account that accepted the Gemma 3 licence). Only part 1 needs it.
3. **Internet: On.**
4. Part 1 needs **GPU T4 x2**. Part 2 needs no GPU: to save GPU hours, run it as a separate session with `RUN_GEN = False`, `RUN_BM25 = True` and **Accelerator: None**. It downloads about 10 GB, plus Java and Pyserini.

## Output

The results go in `/kaggle/working/p2_latency/`, which is also zipped as `p2_latency.zip`. Download the zip and unzip it into the repo's `runs/latency/`:

- `gen_stages.jsonl`, `gen_summary.csv`: per-question timings and p50 / p95 / mean per condition
- `bm25_search.jsonl`, `bm25_summary.csv`: the same for BM25 search
- `latency_p2.md`: the summary, with the gold time scaled to the L4

In [ ]:
RUN_GEN = True           # part 1: generation latency (GPU T4 x2)
RUN_BM25 = False         # part 2: BM25 search latency (CPU; ~10 GB download)
FILES = ["nq_test", "triviaqa_test"]   # same files as Gopal's latency run
N_GEN = 100              # questions per file for part 1 (only questions with a gold passage)
N_BM25 = 200             # questions per file for part 2
WARMUP = 3

# Gopal's L4 closed-book and BM25-RAG times (ms), results/sagemaker/latency/stages_summary.csv on the aws branch
L4 = {"closed": {"p50": 201.1, "p95": 399.9}, "bm25": {"p50": 229.9, "p95": 513.3}}

import json, os, shutil, subprocess, sys, time
from pathlib import Path

INPUT, WORK, SCRATCH = Path("/kaggle/input"), Path("/kaggle/working"), Path("/tmp/saerag_lat")
OUT = WORK / "p2_latency"
OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# The shared code and data from the bundle, as in the labelling notebook.
found = sorted(INPUT.rglob("BUNDLE_INFO.json"))
assert len(found) == 1, f"attach saerag-p2-bundle as an input dataset (found {len(found)} copies)"
BUNDLE = found[0].parent
print("bundle:", json.loads(found[0].read_text()))
REPO, DATA = SCRATCH / "repo", SCRATCH / "data"
shutil.rmtree(SCRATCH, ignore_errors=True)
shutil.copytree(BUNDLE / "src", REPO / "src")
shutil.copytree(BUNDLE / "configs", REPO / "configs")
shutil.copytree(BUNDLE / "data", DATA)

if sys.version_info < (3, 12):
    schema = REPO / "src/saerag/schema.py"
    src = schema.read_text()
    for old, new in [("def _from_dict[T](cls: type[T], d: dict) -> T:", "def _from_dict(cls, d):"),
                     ("def read_jsonl[T](path: str | Path, cls: type[T] | None = None) -> Iterator[T]:",
                      "def read_jsonl(path, cls=None):")]:
        assert old in src, old
        src = src.replace(old, new)
    schema.write_text(src)

os.environ["SAERAG_DATA"] = str(DATA)
sys.path.insert(0, str(REPO / "src"))
from saerag.config import load_config, path
from saerag.schema import Question, read_jsonl

cfg = load_config(REPO / "configs/default.yaml")
GEN, MODEL = cfg["generation"], cfg["model"]["generator"]
assert GEN["max_new_tokens"] == 32 and GEN["temperature"] == 0.0
QS = {f: list(read_jsonl(path(cfg, "questions", f"{f}.jsonl"), Question)) for f in FILES}
print({f: len(q) for f, q in QS.items()})


def summarise(rows, cols):
    import pandas as pd
    df = pd.DataFrame(rows)
    return pd.DataFrame({"p50": df[cols].median(), "p95": df[cols].quantile(0.95),
                         "mean": df[cols].mean(), "n": df[cols].count()}).round(1)

## Part 1: generation latency per condition (GPU)

In [ ]:
# Install the team's transformers / accelerate pins WITHOUT touching Kaggle's torch, and check the
# result in a fresh process before this kernel imports torch. Replacing torch's files after torch
# is loaded leaves a half-old, half-new torch ("module 'torch' has no attribute '_utils'").
if RUN_GEN:
    from importlib.metadata import version
    if "torch" in sys.modules:
        raise RuntimeError("torch is already loaded in this session. Use Run > Restart & clear "
                           "cell outputs, then Run All, so the install happens before torch loads.")
    PINS = {"transformers": "5.19.0", "accelerate": "1.15.0"}
    TORCH = version("torch")
    Path("/tmp/keep_torch.txt").write_text(f"torch=={TORCH}\n")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/keep_torch.txt",
                        *(f"{k}=={v}" for k, v in PINS.items())], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    assert r.returncode == 0, "pip failed; read its output above"
    check = subprocess.run([sys.executable, "-c",
                            "import torch, transformers, accelerate; "
                            "from transformers import AutoModelForCausalLM; "
                            "print(torch.__version__, transformers.__version__, accelerate.__version__, "
                            "torch.cuda.is_available())"], capture_output=True, text=True)
    print("fresh-process check:", check.stdout.strip(), check.stderr[-1500:])
    assert check.returncode == 0, "torch + transformers do not import together; see the error above"
    assert version("torch") == TORCH, f"torch changed from {TORCH} to {version('torch')}"

In [ ]:
if RUN_GEN:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("no HF_TOKEN secret:", e)

    import torch
    from saerag.generation.run import build_messages
    from saerag.hf import chat_prompt, first_line, load_model
    from saerag.answers import is_correct

    assert torch.cuda.is_available(), "part 1 needs Accelerator: GPU T4 x2"
    tok, model, dev = load_model(MODEL, GEN.get("dtype", "auto"))
    GPU = f"{torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}, {model.dtype}"
    print(GPU)

In [ ]:
if RUN_GEN:
    def sync():
        for i in range(torch.cuda.device_count()):
            torch.cuda.synchronize(i)

    def run(msgs, n_tokens):
        enc = tok(chat_prompt(tok, msgs), return_tensors="pt", add_special_tokens=False).to(dev)
        with torch.no_grad():
            sync(); t = time.perf_counter()
            out = model.generate(**enc, max_new_tokens=n_tokens, do_sample=False)
            sync()
        ms = (time.perf_counter() - t) * 1000
        new = out[0, enc.input_ids.shape[1]:]
        return ms, enc.input_ids.shape[1], len(new), tok.decode(new, skip_special_tokens=True)

    CONDS = ("closed", "bm25", "gold")
    gen_rows = []
    for f in FILES:
        msgs = {}
        for c in CONDS:
            qs, ms = build_messages(cfg, c, f)
            msgs[c] = {q.qid: m for q, m in zip(qs, ms)}
        both = [q for q in QS[f] if all(q.qid in msgs[c] for c in CONDS)][:N_GEN]
        for q in both[:WARMUP]:
            for c in CONDS:
                run(msgs[c][q.qid], 4)
        for q in both:
            row = {"qid": q.qid, "file": f}
            for c in CONDS:
                ms, n_in, n_out, text = run(msgs[c][q.qid], GEN["max_new_tokens"])
                row.update({f"{c}_ms": ms, f"{c}_in_tokens": n_in, f"{c}_out_tokens": n_out,
                            f"{c}_correct": is_correct(first_line(text), q.answers)})
            gen_rows.append(row)
        print(f"{f}: timed {len(both)} questions x {len(CONDS)} conditions", flush=True)

    with open(OUT / "gen_stages.jsonl", "w") as fh:
        fh.writelines(json.dumps(r) + "\n" for r in gen_rows)
    cols = [f"{c}_ms" for c in CONDS]
    gen_summary = summarise(gen_rows, cols)
    gen_summary.to_csv(OUT / "gen_summary.csv")
    print(gen_summary.to_string())
    import pandas as pd
    g = pd.DataFrame(gen_rows)
    print("mean input tokens:", {c: round(g[f"{c}_in_tokens"].mean()) for c in CONDS})
    print("accuracy on these questions:", {c: round(g[f"{c}_correct"].mean(), 3) for c in CONDS})

## Part 2: BM25 search latency (CPU)

Pyserini needs Java 21. The prebuilt index goes to `/tmp`, because `/kaggle/working` is limited to 20 GB.

In [ ]:
if RUN_BM25:
    def sh(cmd):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print(r.stdout[-1500:], r.stderr[-1500:])
        return r.returncode

    if sh("java -version 2>&1 | grep -q 'version \"2[1-9]'") != 0:
        sh("apt-get update -qq && apt-get install -y -qq openjdk-21-jdk-headless")
    jhome = subprocess.run("dirname $(dirname $(readlink -f $(which java)))", shell=True,
                           capture_output=True, text=True).stdout.strip()
    os.environ["JAVA_HOME"] = jhome
    print("JAVA_HOME", jhome)
    assert sh("pip install -q pyserini") == 0
    os.environ["PYSERINI_CACHE"] = "/tmp/pyserini"
    print("free in /tmp:", shutil.disk_usage("/tmp").free // 2**30, "GB")

In [ ]:
if RUN_BM25:
    from saerag.retrieval.bm25 import BM25Tool

    t0 = time.time()
    tool = BM25Tool(cfg["retrieval"]["bm25_index"], threads=1)   # downloads the index the first time
    print(f"index ready in {(time.time() - t0) / 60:.1f} min;",
          "documents:", tool.searcher.num_docs)
    assert tool.searcher.num_docs == 21015324, "not the 21,015,324-passage DPR Wikipedia corpus"

    def timed_search(q, k):
        t = time.perf_counter()
        hits = tool.search(q, k)
        return (time.perf_counter() - t) * 1000, len(hits)

    K_DEPLOY, K_FETCH = cfg["retrieval"]["top_k"], cfg["retrieval"]["hole_fetch_k"]
    bm25_rows = []
    for f in FILES:
        qs = QS[f][:N_BM25]
        for q in qs[:20]:                        # warm the page cache and the JVM
            tool.search(q.question, K_DEPLOY)
        for q in qs:
            ms3, n3 = timed_search(q.question, K_DEPLOY)
            ms100, _ = timed_search(q.question, K_FETCH)
            bm25_rows.append({"qid": q.qid, "file": f, f"search_k{K_DEPLOY}_ms": ms3,
                              f"search_k{K_FETCH}_ms": ms100, "hits": n3})
        print(f"{f}: searched {len(qs)} questions", flush=True)

    with open(OUT / "bm25_search.jsonl", "w") as fh:
        fh.writelines(json.dumps(r) + "\n" for r in bm25_rows)
    bm25_summary = summarise(bm25_rows, [f"search_k{K_DEPLOY}_ms", f"search_k{K_FETCH}_ms"])
    bm25_summary.to_csv(OUT / "bm25_summary.csv")
    print(bm25_summary.to_string())

## Summary

In [ ]:
lines = ["# Person 2 latency at batch size 1", ""]
if RUN_GEN:
    lines += [f"## Generation per condition ({GPU}, files {', '.join(FILES)}, "
              f"{len(gen_rows)} questions with a gold passage)", "",
              "```", gen_summary.to_string(), "```", ""]
    ratio = {s: gen_summary.loc["gold_ms", s] / gen_summary.loc["bm25_ms", s] for s in ("p50", "p95")}
    lines += [f"gold / BM25-RAG time ratio on this GPU: p50 {ratio['p50']:.2f}, p95 {ratio['p95']:.2f}", "",
              "This ratio does not carry over to the L4: in fp32 on a T4 the prompt (prefill) dominates, "
              "while on the L4 in bf16 BM25-RAG costs only ~1.14x closed-book.", ""]
    # Generation time is closed-book time plus a prefill cost that grows with prompt length, so place
    # gold between Gopal's L4 closed and BM25-RAG times by its mean prompt length.
    tokens = {c: pd.DataFrame(gen_rows)[f"{c}_in_tokens"].mean() for c in CONDS}
    frac = (tokens["gold"] - tokens["closed"]) / (tokens["bm25"] - tokens["closed"])
    est = {s: L4["closed"][s] + frac * (L4["bm25"][s] - L4["closed"][s]) for s in ("p50", "p95")}
    lines += [f"Mean prompt tokens: closed {tokens['closed']:.0f}, gold {tokens['gold']:.0f}, "
              f"BM25 {tokens['bm25']:.0f} (gold sits {frac:.0%} of the way from closed to BM25).", "",
              f"**Gold generation on the L4 (estimate): p50 {est['p50']:.1f} ms, p95 {est['p95']:.1f} ms**", ""]
if RUN_BM25:
    lines += [f"## BM25 search, one query at a time, one thread "
              f"({len(bm25_rows)} questions, {os.cpu_count()} CPUs)", "",
              "```", bm25_summary.to_string(), "```", "",
              f"A routed RETRIEVE costs BM25 search (k = {K_DEPLOY}) + RAG generation.", ""]
text = "\n".join(lines)
(OUT / "latency_p2.md").write_text(text)
print(text)
shutil.make_archive(str(WORK / "p2_latency"), "zip", OUT)
print("download /kaggle/working/p2_latency.zip from the Output tab")